# 04 — Churn Prediction and CLV Estimation

The centerpiece of the project: a data-driven churn window feeds a churn model, whose output feeds a CLV formula and a CLV regressor, compared against each other.

**Leakage note**: `segment_id` is deliberately excluded from both churn and CLV model features. Phase 3's segmentation clustered on `recency_days`/`monetary` directly, so including `segment_id` would leak the label (churn is a recency threshold; CLV's target `monetary` was a direct clustering input) through cluster membership. This was caught empirically: an earlier run WITH segment_id produced a suspicious 0.98 ROC-AUC; removing it dropped to a realistic 0.76.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb pandas pyarrow scikit-learn xgboost lightgbm mlxtend shap prophet google-genai

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DIR / "features"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse.duckdb"

for d in [PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence


## Churn window: derived from data, not guessed

In [2]:

import duckdb, pandas as pd

con = duckdb.connect(str(WAREHOUSE_PATH))
gap_query = """
SELECT c.customer_unique_id, d.full_date AS order_date
FROM fact_order_item f
JOIN dim_customer c ON f.customer_key = c.customer_key
JOIN dim_date d ON f.order_date_key = d.date_key
"""
order_dates = con.execute(gap_query).df()
con.close()
order_dates["order_date"] = pd.to_datetime(order_dates["order_date"])
order_dates = order_dates.drop_duplicates(subset=["customer_unique_id", "order_date"]).sort_values(
    ["customer_unique_id", "order_date"])
order_dates["gap_days"] = order_dates.groupby("customer_unique_id")["order_date"].diff().dt.days
gaps = order_dates["gap_days"].dropna()

print(f"{len(gaps):,} inter-purchase gaps from repeat customers")
print(f"median={gaps.median():.1f}  p75={gaps.quantile(.75):.1f}  p90={gaps.quantile(.90):.1f}  p95={gaps.quantile(.95):.1f}")

CHURN_WINDOW_DAYS = int(round(gaps.quantile(0.90)))
print(f"\nChurn window: {CHURN_WINDOW_DAYS} days (p90 of repeat-purchase gaps)")


2,309 inter-purchase gaps from repeat customers
median=69.0  p75=169.0  p90=280.2  p95=342.6

Churn window: 280 days (p90 of repeat-purchase gaps)


## Build churn features and labels

In [3]:

customer_features = pd.read_parquet(FEATURES_DIR / "customer_features.parquet")
segments = pd.read_parquet(MODELS_DIR / "customer_segments.parquet")[["customer_unique_id", "segment_id"]]
df = customer_features.merge(segments, on="customer_unique_id", how="inner")

df["churned"] = (df["recency_days"] > CHURN_WINDOW_DAYS).astype(int)
print(f"Churn rate: {df['churned'].mean()*100:.1f}% ({df['churned'].sum():,} / {len(df):,})")

NULLABLE_FEATURES = ["avg_review_score", "avg_delivery_days", "avg_delivery_delay"]

def build_feature_matrix(df, drop_extra=()):
    drop_cols = ["customer_unique_id", "recency_days", "first_purchase", "last_purchase",
                 "churned", "segment_id"] + list(drop_extra)
    X = df[[c for c in df.columns if c not in drop_cols]].copy()
    for col in NULLABLE_FEATURES:
        X[f"{col}_missing"] = X[col].isna().astype(int)
        X[col] = X[col].fillna(X[col].median())
    X = pd.get_dummies(X, columns=["customer_state"], prefix="state")
    return X

X_churn = build_feature_matrix(df)
y_churn = df["churned"]
print(f"Churn feature matrix: {X_churn.shape}")


Churn rate: 38.2% (36,468 / 95,420)
Churn feature matrix: (95420, 40)


## Train and compare churn models

In [4]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_churn, y_churn, df.index, test_size=0.2, random_state=42, stratify=y_churn)

scaler = StandardScaler()
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(scaler.fit_transform(X_train), y_train)
logreg_proba = logreg.predict_proba(scaler.transform(X_test))[:, 1]

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb_churn = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                           scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42)
xgb_churn.fit(X_train, y_train)
xgb_proba = xgb_churn.predict_proba(X_test)[:, 1]

def metrics(y_true, proba, thresh=0.5):
    pred = (proba >= thresh).astype(int)
    return dict(precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred, zero_division=0),
                f1=f1_score(y_true, pred, zero_division=0),
                roc_auc=roc_auc_score(y_true, proba))

print("Logistic Regression:", metrics(y_test, logreg_proba))
print("XGBoost:            ", metrics(y_test, xgb_proba))

winner_proba_full = xgb_churn.predict_proba(X_churn)[:, 1]  # XGBoost expected to win; verify against printed metrics
df["churn_probability"] = winner_proba_full
print(f"\nFull-population mean churn_probability: {df['churn_probability'].mean():.3f}")


Logistic Regression: {'precision': 0.4379420488250057, 'recall': 0.5263230052097615, 'f1': 0.4780821917808219, 'roc_auc': np.float64(0.5763413141455221)}
XGBoost:             {'precision': 0.5605062231984103, 'recall': 0.7347134631203729, 'f1': 0.6358943933550875, 'roc_auc': np.float64(0.7635100468322692)}

Full-population mean churn_probability: 0.479


## CLV: formula-based vs. ML regressor

`CLV = avg_order_value * frequency * (1/churn_probability)`, churn_probability clipped to [0.02, 0.98] to avoid unbounded estimates. The ML regressor predicts realized historical spend (`monetary`) -- a backward-looking target, unlike the formula's forward-looking projection, so they're evaluated differently (RMSE/R² for the regressor; top-decile rank overlap for the formula -- see methodology notes).

In [5]:

CLIP_MIN, CLIP_MAX = 0.02, 0.98
clipped = df["churn_probability"].clip(CLIP_MIN, CLIP_MAX)
df["clv_formula"] = df["avg_order_value"] * df["frequency"] * (1 / clipped)
print(f"Formula CLV: mean={df['clv_formula'].mean():.2f}  median={df['clv_formula'].median():.2f}")


Formula CLV: mean=485.94  median=262.31


In [6]:

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

X_clv = build_feature_matrix(df, drop_extra=["monetary"])
X_clv["churn_probability"] = df["churn_probability"].values
y_clv = df["monetary"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_clv, y_clv, test_size=0.2, random_state=42)

xgb_clv = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42)
xgb_clv.fit(Xc_train, yc_train)
xgb_clv_pred = xgb_clv.predict(Xc_test)

lgbm_clv = LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
lgbm_clv.fit(Xc_train, yc_train)
lgbm_clv_pred = lgbm_clv.predict(Xc_test)

def reg_metrics(y_true, pred):
    return dict(rmse=mean_squared_error(y_true, pred) ** 0.5,
                mae=mean_absolute_error(y_true, pred), r2=r2_score(y_true, pred))

print("XGBoost CLV: ", reg_metrics(yc_test, xgb_clv_pred))
print("LightGBM CLV:", reg_metrics(yc_test, lgbm_clv_pred))

# LightGBM typically wins narrowly -- verify against printed metrics before trusting this line
df["clv_ml"] = lgbm_clv.predict(X_clv)
print(f"\nFull-population mean clv_ml: {df['clv_ml'].mean():.2f}  mean clv_formula: {df['clv_formula'].mean():.2f}")


XGBoost CLV:  {'rmse': 65.06757251961335, 'mae': 3.0516091881485963, 'r2': 0.9276999885097937}
LightGBM CLV: {'rmse': 64.29325344974714, 'mae': 3.1771486748522912, 'r2': 0.929410523221851}

Full-population mean clv_ml: 165.97  mean clv_formula: 485.94


## Save predictions

In [7]:

churn_predictions = df[["customer_unique_id", "segment_id", "churn_probability"]].copy()
churn_predictions.to_parquet(MODELS_DIR / "churn_predictions.parquet", index=False)

clv_predictions = df[["customer_unique_id", "segment_id", "churn_probability", "clv_ml", "clv_formula", "monetary"]].copy()
clv_predictions.to_parquet(MODELS_DIR / "clv_predictions.parquet", index=False)

print("Saved churn_predictions.parquet and clv_predictions.parquet")


Saved churn_predictions.parquet and clv_predictions.parquet
